<a href="https://colab.research.google.com/github/samrat-dhakal-11/Google-ADK-Agent/blob/main/Agent_Team_google_adk.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Agent Team

In [ ]:

# @title Step 0: Setup and Installation
# Install ADK and LiteLLM for multi-model support

!pip install google-adk -q
!pip install "litellm>=1.84" -q

print("Installation complete.")

Installation complete.


In [19]:
import os
import asyncio
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.genai import types

import warnings
warnings.filterwarnings("ignore")

import logging
logging.basicConfig(level=logging.ERROR)

import requests
from google.colab import userdata
openai_key=userdata.get("OPENAI_API_KEY")
gemini_key=userdata.get("GEMINI_API_KEY")
weather_api_key=userdata.get("WEATHER_API_KEY")

In [ ]:
os.environ["GOOGLE_API_KEY"] = openai_key
os.environ['OPENAI_API_KEY'] = gemini_key
os.environ['GOOGLE_GENAI_USE_ENTERPISE']="False"

In [ ]:
Gemini_Model="gemini-3.1-flash-lite"
Openai_Model="openai/gpt-5.4-mini"

In [34]:
def get_weather(city:str):

  """ Get all current weather information for a city.

  Args:city(str) Recives name of the city

  Returns:
        Status and
        The full WeatherAPI response, including location (name, region,
        country, lat, lon, local time) and current weather (temperature,
        condition, wind, humidity, rain, and more).

  """


  try:
    res=requests.get(f'https://api.weatherapi.com/v1/current.json?key={weather_api_key}&q={city}')
    data=res.json()

  except Exception as e:
    return {'status':"failed","msg":str(e)}

  if res.status_code != 200:
    return {"status": "error", "msg": data.get("error", {}).get("message", "Unknown error")}

  return {"status": "success", "msg": data}

In [35]:
Agent_Model=LiteLlm(model=Openai_Model)

In [36]:
weather_agent=Agent(
 model=Agent_Model,
 name="Weather_Agent",
 description="Provides Weather Information for all cities in world",
 instruction="You are a helpful weather assistant. "
             "When the user asks for the weather of cities, "
             "use the 'get_weather' tool to find the information. "
             "If the tool returns an error, inform the user politely. "
             "If the tool is successful, present the weather report clearly.",
 tools=[get_weather],
)